## Import necessary packages 

In [1]:
# Python modules 
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.rcParams.update(matplotlib.rcParamsDefault)
import pandas as pd
import importlib
from numpy import loadtxt
from matplotlib import gridspec # for the contour plots
from cvxopt import matrix, solvers

# pyDRTtools' modules
import pyDRTtools
import pyDRTtools.basics as basics # pyDRTtools functions
import pyDRTtools.GUI as UI

import os
from glob import glob

Initializing pyDRTtools from C:\Users\Herman\Python\PythonEnvs\py314_env\Lib\site-packages
['', 'C:\\Users\\Herman\\psi4conda\\Lib\\site-packages', 'C:\\Users\\Herman\\psi4conda\\Lib\\site-packages\\_plotly_utils', 'C:\\Users\\Herman\\.spyder-py3', 'C:\\Users\\Herman\\Python\\pythoncore-3.14-64\\python314.zip', 'C:\\Users\\Herman\\Python\\pythoncore-3.14-64\\DLLs', 'C:\\Users\\Herman\\Python\\pythoncore-3.14-64\\Lib', 'C:\\Users\\Herman\\Python\\pythoncore-3.14-64', 'C:\\Users\\Herman\\Python\\PythonEnvs\\py314_env', 'C:\\Users\\Herman\\Python\\PythonEnvs\\py314_env\\Lib\\site-packages']


C:\Users\Herman\Python\PythonEnvs\py314_env\Lib\site-packages\pyDRTtools\GUI.py:347: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  'k', label = '$Z_\mu$(Regressed)', linewidth=3)
C:\Users\Herman\Python\PythonEnvs\py314_env\Lib\site-packages\pyDRTtools\GUI.py:360: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  self.axes.set_xlabel('$Z^{\prime}/\Omega$')
C:\Users\Herman\Python\PythonEnvs\py314_env\Lib\site-packages\pyDRTtools\GUI.py:361: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  self.axes.set_ylabel('-$Z^{\prime \prime}/\Omega$')
C:\Users\Herman\Python\PythonEnvs\py314_env\Lib\site-packages\pyDRTtools\GUI.py:369: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences wil

ModuleNotFoundError: No module named 'PyQt5'

In [ ]:
## for nice plot
# options for the plots
plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=15)
plt.rc('xtick', labelsize=15)
plt.rc('ytick', labelsize=15)
#%matplotlib inline

### 1. Analysis of a single EIS spectrum generated with the Cole-Cole model

#### 1.1 Load the data

In [11]:
s239 = {'name' : '239', 'comp' : '0 O2','%' : 0.33}
MEA = [s239]

for mea in MEA:
    mea['folder']= glob(os.path.join('//ELECTROLYZER/PEM-WE_measurements/2025/'+'*'+mea['name']+'*'))
    mea['files'] =  glob(os.path.join(mea['folder'][0],'*rocedur*'+'*PEIS*'+'*.mpt'))
    
    print(mea['files'])
    
def get_header_lines(filepath):
    with open(filepath, "r", encoding="latin-1") as f:
        # Skip first line, read second
        f.readline()  
        line2 = f.readline().strip()
    # line2 looks like: "Nb header lines : 76"
    # split by ":" and take the right part
    try:
        n_header = int(line2.split(":")[1])
    except Exception:
        raise ValueError(f"Could not parse header line: {line2}")
    return n_header

['//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day1_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day2_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day3_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day4_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day5_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day6_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day7_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER/PEM-WE_measurements/2025\\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\\IV_Day8_procedure1_05_PEIS_C01.mpt', '//ELECTROLYZER

In [12]:
file = s239['files'][0]
print(file)
skip_rows = get_header_lines(file)

with open(file, encoding='latin1') as f:
    df = pd.read_csv(f,skiprows = skip_rows-1,usecols=np.arange(0,60),delimiter = '\t')
N_freqs = df.shape[0]
print(N_freqs)
freq_vec = (df.loc[df["cycle number"] == 8, 'freq/Hz'].values)
Z_exp = (df.loc[df["cycle number"] == 8, 'Re(Z)/Ohm'].values - 1j*df.loc[df["cycle number"] == 8, '-Im(Z)/Ohm'].values)
# Z_exp = np.flip(df['Real'].values + 1j*df['Imag'].values)
Z_exp = Z_exp[13:]
freq_vec = freq_vec[13:]
N_freqs = len(freq_vec)
print(Z_exp)
print(freq_vec)

//ELECTROLYZER/PEM-WE_measurements/2025\239_IV_IV IrOx series 0 sccm O2 15 sccm Ar\IV_Day1_procedure1_05_PEIS_C01.mpt
5379
[0.00052752+0.05976977j 0.0042041 +0.05125819j 0.00684474+0.04412205j
 0.00846392+0.03790893j 0.00964389+0.03252911j 0.01052177+0.02790852j
 0.01130964+0.02409043j 0.01182619+0.02105405j 0.01192403+0.01866641j
 0.01160185+0.01648211j 0.01119284+0.01435842j 0.01082319+0.01234123j
 0.01055817+0.01053427j 0.01037698+0.00886604j 0.01026851+0.00739683j
 0.01019815+0.00607655j 0.01022284+0.00491955j 0.01026873+0.00390324j
 0.01027166+0.00302468j 0.01031991+0.00222788j 0.01038583+0.00149083j
 0.01046185+0.00079755j 0.01052568+0.00013811j 0.01062815-0.00047091j
 0.01076071-0.00109769j 0.01090012-0.00171251j 0.01109114-0.00235357j
 0.01127155-0.00301053j 0.01156008-0.00372413j 0.01185429-0.00443153j
 0.01218009-0.0051988j  0.01261329-0.00603261j 0.0131441 -0.00692404j
 0.01376097-0.00787978j 0.01450484-0.0088651j  0.01541663-0.00993663j
 0.01650942-0.01100546j 0.01780037-0.

#### 1.2 Define the range of timescales

In [3]:
N_taus = N_freqs
log_tau_min = -4  
log_tau_max = 0  
tau_vec = np.logspace(log_tau_min, log_tau_max, num = N_taus, endpoint=True)
log_tau_vec = np.log(tau_vec)

NameError: name 'N_freqs' is not defined

#### 1.3 Define the discretization matrices

In [14]:
shape_control = 'FWHM Coefficient'
coeff = 0.5
# order of the derivative for the differentiation matrix M ('1st', '2nd')
# rbf_type = 'Piecewise Linear'
rbf_type = 'Gaussian'
# rbf_type = 'C0 Matern'
# rbf_type = 'C2 Matern'
# rbf_type = 'C4 Matern'
# rbf_type = 'C6 Matern'
# rbf_type = 'Inverse Quadratic'


In [15]:
cv_type = 'GCV'
# cv_type = 'mGCV'
# cv_type = 'rGCV'
# cv_type = 'LC'
# cv_type = 'kf'
# cv_type = 're-im'

#### 1.4 Compute the discretization matrices

In [16]:
# epsilon parameter
epsilon  = basics.compute_epsilon(freq_vec, coeff, rbf_type, shape_control)

# differentiation matrix for the real part of the impedance
# this includes the contributions of an Ohmic resistance, R_inf, and an inductance, L_0
A_re = basics.assemble_A_re(freq_vec, tau_vec, epsilon, rbf_type)

A_re_R_inf = np.ones((N_freqs, 1))
A_re_L_0 = np.zeros((N_freqs, 1))
A_re = np.hstack(( A_re_R_inf, A_re_L_0, A_re))

# differentiation matrix for the imaginary part of the impedance
A_im = basics.assemble_A_im(freq_vec, tau_vec, epsilon, rbf_type)

#assemble_A_im(freq_vec, tau_vec, epsilon, 'Piecewise Linear', flag1='simple', flag2='impedance')

A_im_R_inf = np.zeros((N_freqs, 1))
A_im_L_0 = 2*np.pi*freq_vec.reshape((N_freqs, 1))
A_im = np.hstack(( A_im_R_inf, A_im_L_0, A_im))

# complete discretization matrix
A = np.vstack((A_re, A_im))

#### 1.5 Define the differentiation matrix

In [17]:
# second-order differentiation matrix for ridge regression (RR)
M2 = np.zeros((N_taus+2, N_taus+2))
M2[2:,2:] = basics.assemble_M_2(tau_vec, epsilon, rbf_type)

#### 1.6 Select the regularization parameter

In [18]:
# regularization parameter obtained with generalized cross-validation (GCV)
lambda_value = basics.optimal_lambda(
    A_re, A_im,
    np.real(Z_exp), np.imag(Z_exp),
    M2,
    'both',   # use both Re and Im data
    'no',     # no inductive part handling
    -3,       # starting log10(lambda) (the code will scan from here)
    'GCV'     # generalized cross-validation
)
print(lambda_value)

Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.5564366065706974e-06
            Iterations: 1
            Function evaluations: 2
            Gradient evaluations: 1
GCV
[0.04978707]


#### 1.7 Deconvolve the DRT from a single EIS spectrum

In [19]:

# we recover the DRT as a quadratic problem using cvxplot
lb = np.zeros([N_taus+2])
bound_mat = np.eye(lb.shape[0])

H_combined, c_combined = basics.quad_format_combined(A_re, A_im, np.real(Z_exp), np.imag(Z_exp), M2, lambda_value)

# set bound constraint
G = matrix(-np.identity(Z_exp.imag.shape[0]+2))
print(len(G[:,0]))

h = matrix(np.zeros(Z_exp.imag.shape[0]+2))
sol = solvers.qp(matrix(H_combined), matrix(c_combined),G,h)
## deconvolved DRT
x = np.array(sol['x']).flatten()

R_inf_DRT, L_0_DRT = x[0:2]
gamma_DRT = x[2:]


## plot the recovered DRT

# # fig = plt.gcf()
# plt.semilogx(tau_vec, gamma_DRT, linewidth=4, color='black', label='DRT') 
# plt.axis([1E-6, 1E2, 0, 1])
# plt.legend(frameon=False, fontsize = 15, loc='upper left')
# plt.xlabel(r'$\tau/\rm s$', fontsize = 20)
# plt.ylabel(r'$\gamma/\Omega$', fontsize = 20)
# fig.set_size_inches(6.472, 4)

# plt.show()

83
     pcost       dcost       gap    pres   dres
 0: -1.0636e-01 -2.4093e-01  8e+01  9e+00  3e-05
 1:  1.4916e+00 -2.3651e+00  7e+00  4e-01  1e-06
 2:  3.6141e-01 -7.1435e-01  1e+00  4e-02  1e-07
 3: -2.7666e-02 -2.2139e-01  2e-01  4e-04  1e-09
 4: -8.1992e-02 -1.6643e-01  8e-02  3e-06  1e-11
 5: -1.0230e-01 -1.1255e-01  1e-02  2e-07  5e-13
 6: -1.0577e-01 -1.0742e-01  2e-03  1e-17  8e-17
 7: -1.0629e-01 -1.0652e-01  2e-04  6e-18  1e-16
 8: -1.0636e-01 -1.0639e-01  3e-05  5e-18  2e-19
 9: -1.0637e-01 -1.0637e-01  6e-06  4e-18  6e-17
10: -1.0637e-01 -1.0637e-01  8e-07  3e-18  1e-16
11: -1.0637e-01 -1.0637e-01  1e-07  4e-18  3e-17
Optimal solution found.


In [20]:
## plot the recovered DRT

###&#####fig = plt.gcf()
plt.semilogx(tau_vec, gamma_DRT, linewidth=4, color='black', label='DRT') 
plt.axis([1E-5, 1E1, 0, 0.075])
plt.legend(frameon=False, fontsize = 15, loc='upper left')
#plt.xlabel(r'$\tau/\rm s$', fontsize = 20)
#plt.ylabel(r'$\gamma/\Omega$', fontsize = 20)
#fig.set_size_inches(6.472, 4)

plt.show()

#### 1.8 Recover the impedance

In [197]:
# the impedance is recovered as a matrix product
Z_DRT = A@x
Z_DRT = Z_DRT[0:N_freqs] + 1j*Z_DRT[N_freqs:]

#### 1.9 Nyquist plots

In [200]:
# Nyquist plots of the exact, experimental, and recovered DRTs
plt.plot(np.real(Z_DRT), -np.imag(Z_DRT), linewidth=4, color='black', label='R')
plt.plot(np.real(Z_exp), -np.imag(Z_exp), 'o', markersize=7, color='red', label='exp')
plt.legend(frameon=False, fontsize = 15)
plt.axis('scaled')
#plt.xticks(range(0, 90, 20))
#plt.yticks(range(-40, 60, 20))
plt.gca().set_aspect('equal', adjustable='box')
plt.xlabel(r'$Z_{\rm re}/\Omega$', fontsize = 20)
plt.ylabel(r'$-Z_{\rm im}/\Omega$', fontsize = 20)
fig = plt.gcf()
fig.set_size_inches(6.472, 5)

plt.show()

#### 1.10 Save the recovered DRT and impedance

In [221]:
# save the DRT recovered with RR
gamma_DRT_excel = pd.DataFrame(gamma_DRT)
gamma_DRT_excel.to_csv('./results/1ZARC_DRT-RR.csv',index=False)

# save the impedance recovered with RR
df = pd.DataFrame.from_dict({'Freq': freq_vec, 'Real': np.real(Z_DRT), 'Imag': np.imag(Z_DRT)})
df.to_csv('./results/1ZARC_Z-RR.csv')

### 2. Analysis of multiple EIS spectra

#### 2.1 Load and analyze many EIS spectra concomitantly

In [37]:
s244 = {'name' : '244', 'comp' : '30 Ir 30 C'}
MEA = [s244]

for mea in MEA:
    # mea['folder']= glob(os.path.join('//ELECTROLYZER/PEM-WE_measurements/2025/'+'*'+mea['name']+'*'))
    # mea['files'] =  glob(os.path.join(mea['folder'][0],'*rocedur*'+'*PEIS*'+'*.mpt'))
    folder = r"C:/Users/Herman/Desktop/"
    mea['files'] =  glob(os.path.join(folder,'*batch2_3_batch1_interruptions_1,56V_01_PEIS_C01.mpt'))
    
    
    print(mea['files'])
    
def get_header_lines(filepath):
    with open(filepath, "r", encoding="latin-1") as f:
        # Skip first line, read second
        f.readline()  
        line2 = f.readline().strip()
    # line2 looks like: "Nb header lines : 76"
    # split by ":" and take the right part
    try:
        n_header = int(line2.split(":")[1])
    except Exception:
        raise ValueError(f"Could not parse header line: {line2}")
    return n_header

['C:/Users/Herman/Desktop\\batch2_3_batch1_interruptions_1,56V_01_PEIS_C01.mpt']


In [22]:
shape_control = 'FWHM Coefficient'
coeff = 0.5
# order of the derivative for the differentiation matrix M ('1st', '2nd')
# rbf_type = 'Piecewise Linear'
rbf_type = 'Gaussian'
# rbf_type = 'C0 Matern'
# rbf_type = 'C2 Matern'
# rbf_type = 'C4 Matern'
# rbf_type = 'C6 Matern'
# rbf_type = 'Inverse Quadratic'

In [23]:
cv_type = 'GCV'
# cv_type = 'mGCV'
# cv_type = 'rGCV'
# cv_type = 'LC'
# cv_type = 'kf'
# cv_type = 're-im'

In [62]:
for file in mea['files']:
    print(file)
#     print((n-n%3)/3)
#     file = s244['files'][int((n-n%3)/3)]
    # file = s244['files'][n]
    skip_rows = get_header_lines(file)
    with open(file, encoding='latin1') as f:
        df = pd.read_csv(f,skiprows = skip_rows-1,usecols=np.arange(0,30),delimiter = '\t')
        
        
    unique_vals = np.unique(df["cycle number"].values)
    # groups = [data[df["cycle number"].values == val] for val in unique_vals]
    
    R_inf_DRT_list = []
    L_0_DRT_list = []
    gamma_DRT_list = []
    # Z_DRT_list = [0]*len(unique_vals)
    # Z_exp_list = [0]*len(unique_vals)
    Z_exp_list = []
    freq_list = []
    log_tau_list = []
    
    for val in unique_vals:
        freq_vec = (df.loc[(df["cycle number"] == val) & (df["freq/Hz"] != 0), 'freq/Hz'].values)
        Z_exp = (df.loc[(df["cycle number"] == val) & (df["freq/Hz"] != 0), 'Re(Z)/Ohm'].values - 1j*df.loc[(df["cycle number"] == val) & (df["freq/Hz"] != 0), '-Im(Z)/Ohm'].values)
        if np.diff(freq_vec).any() <  0:
            print('problem')
        else:
            pass
        print(freq_vec)
        Z_exp_list.append(Z_exp)
        freq_list.append(freq_vec)
        N_freqs = len(freq_vec)
        N_taus = N_freqs
        
        log_tau_min = round(np.log10(1/max(freq_vec)),3)  
        log_tau_max = round(np.log10(1/min(freq_vec)),3)   
        tau_vec = np.logspace(log_tau_min, log_tau_max, num = N_taus, endpoint=True)
        log_tau_vec = np.log10(tau_vec)
        log_tau_list.append(log_tau_vec)
        
        print(log_tau_min,log_tau_max)
        
        epsilon  = basics.compute_epsilon(freq_vec, coeff, rbf_type, shape_control)

        # step 3: compute the differentiation matrices
        A_re = basics.assemble_A_re(freq_vec, tau_vec, epsilon, rbf_type)
        A_re_R_inf = np.ones((N_freqs, 1))
        A_re_L_0 = np.zeros((N_freqs, 1))
        A_re = np.hstack(( A_re_R_inf, A_re_L_0, A_re))
        A_im = basics.assemble_A_im(freq_vec, tau_vec, epsilon, rbf_type)
        A_im_R_inf = np.zeros((N_freqs, 1))
        A_im_L_0 = 2*np.pi*freq_vec.reshape((N_freqs, 1))
        A_im = np.hstack(( A_im_R_inf, A_im_L_0, A_im))
        A = np.vstack((A_re, A_im))

        # step 4: compute the differentiation matrix
        M2 = np.zeros((N_taus+2, N_taus+2))
        M2[2:,2:] = basics.assemble_M_2(tau_vec, epsilon, rbf_type)

        # step 5: compute the regularization parameter
    #     lambda_value = basics.optimal_lambda(A_re, A_im, np.real(Z_exp), np.imag(Z_exp), M2, -3, 'GCV')
        lambda_value = 0.002

        # step 6: recover the DRT
        ### 
        lb = np.zeros([N_taus+2])
        bound_mat = np.eye(lb.shape[0])
        H_combined, c_combined = basics.quad_format_combined(A_re, A_im, np.real(Z_exp), np.imag(Z_exp), M2, lambda_value)
        
        



        # set bound constraint
        G = matrix(-np.identity(Z_exp.imag.shape[0]+2))


        h = matrix(np.zeros(Z_exp.imag.shape[0]+2))

        

        sol = solvers.qp(matrix(H_combined), matrix(c_combined),G,h)
        ## deconvolved DRT
        x = np.array(sol['x']).flatten()

        R_inf_DRT_list.append(x[0])
        L_0_DRT_list.append(x[1])
        gamma_DRT_list.append(np.array(x)[2:])

    
    # save the DRT recovered with RR



        # step 7: recover the impedance
        # Z_DRT = A@x
        # Z_DRT_list[n] = Z_DRT[0:N_freqs] + 1j*Z_DRT[N_freqs:]
        
#     #   


C:/Users/Herman/Desktop\batch2_3_batch1_interruptions_1,56V_01_PEIS_C01.mpt
[6.0000369e+05 3.7657734e+05 2.3635133e+05 1.4834472e+05 9.3109781e+04
 5.8436016e+04 3.6677922e+04 2.3019965e+04 1.4446631e+04 9.0674502e+03
 5.6903809e+03 3.5711431e+03 2.2407166e+03 1.4072515e+03 8.8302197e+02
 5.5446771e+02 3.4788489e+02 2.1839261e+02 1.3697237e+02 8.6018440e+01
 5.3980843e+01 3.3883205e+01 2.1258503e+01 1.3347096e+01 8.3825140e+00
 5.2609429e+00 3.3033826e+00 2.0733812e+00 1.3007827e+00]
-5.778 -0.114
     pcost       dcost       gap    pres   dres
 0: -1.6565e+00 -1.1886e+00  7e+01  8e+00  4e-06
 1: -8.8301e-01 -1.5140e+00  2e+01  2e+00  8e-07
 2:  1.6298e-01 -4.9776e-01  2e+00  2e-01  9e-08
 3:  2.5613e-02 -2.0787e-01  2e-01  2e-03  9e-10
 4: -5.7083e-02 -9.2916e-02  4e-02  2e-05  9e-12
 5: -6.7700e-02 -7.1968e-02  4e-03  3e-07  1e-13
 6: -6.9086e-02 -7.0578e-02  1e-03  3e-18  3e-17
 7: -6.9415e-02 -6.9650e-02  2e-04  2e-18  6e-17
 8: -6.9493e-02 -6.9531e-02  4e-05  3e-18  1e-16
 9: -6.9

In [63]:
def build_spectra_dataframe(energy, spectra, metadata_list=None):
    """
    energy: 1D numpy array
    spectra: list of 1D numpy arrays (all same length as energy)
    metadata_list: list of dicts, same length as spectra
    """

    columns = {}
    
    for i, spec in enumerate(gamma_DRT_list):
        name = f"spec{i+1}"
        columns[(name, "energy")] = energy[i]
        columns[(name, "value")] = spectra[i]
    
    df = pd.DataFrame(columns)

    # Save metadata in df.attrs (not stored by TXT, but stays in memory)
    if metadata_list is not None:
        for i, meta in enumerate(metadata_list):
            df.attrs[f"spec{i+1}_metadata"] = meta

    return df



metadata_list = [
    {"Ro": 5, "bias": 0.1},
    {"T": 7, "bias": -0.05},
    {"T": 4, "bias": 0.2}
]

df = build_spectra_dataframe(log_tau_list, gamma_DRT_list, metadata_list)


df.to_csv(file[:-4]+'_DRT.txt', sep="\t", index=False)

In [48]:
print(file[:-4]+'_DRT.txt')

C:/Users/Herman/Desktop\batch2_3_batch1_interruptions_1,56V_01_PEIS_C01_DRT.txt


In [65]:
# N_files = 6
# N_exp = 3*N_files # number of EIS experiments





# # N_freqs = df.shape[0]
# # print(N_freqs)
# # freq_vec = (df.loc[df["cycle number"] == 8, 'freq/Hz'].values)

# # # Z_exp = np.flip(df['Real'].values + 1j*df['Imag'].values)
# # Z_exp = Z_exp[13:]
# # freq_vec = freq_vec[13:]
# # N_freqs = len(freq_vec)
# # print(Z_exp)
# # print(freq_vec)


# # lists to save the DRTs and impedances
# R_inf_DRT_list = [0]*N_exp
# L_0_DRT_list = [0]*N_exp
# gamma_DRT_list = [0]*N_exp
# Z_DRT_list = [0]*N_exp
# Z_exp_list = [0]*N_exp


# for n in range(0,N_files):
#     print(N_files)
# #     print((n-n%3)/3)
# #     file = s244['files'][int((n-n%3)/3)]
#     file = s244['files'][n]
#     skip_rows = get_header_lines(file)
#     with open(file, encoding='latin1') as f:
#         df = pd.read_csv(f,skiprows = skip_rows-1,usecols=np.arange(0,60),delimiter = '\t')
#     # step 1: load the experimental data
# #     df = pd.read_csv('./data/2ZARC'+str(n)+'.csv') # this artificial data was generated using the 2xZARC model
#     N_freqs = df.shape[0]
#     for i in range(0,3):
#         Z_exp = (df.loc[df["cycle number"] == 5+i*20, 'Re(Z)/Ohm'].values - 1j*df.loc[df["cycle number"] == 5+i*20, '-Im(Z)/Ohm'].values)
#     #     print(Z_exp)
#         freq_vec = (df.loc[df["cycle number"] == 5+i*20, 'freq/Hz'].values)
#         freq_vec = freq_vec[13:]
#         N_freqs = len(freq_vec)
#     #     print(freq_vec)
#         Z_exp = Z_exp[13:]
        
#         Z_exp_list[3*n+i] = Z_exp

# for n in range(0,N_exp):
#     # step 2: compute the epsilon parameter
#     epsilon  = basics.compute_epsilon(freq_vec, coeff, rbf_type, shape_control)

#     # step 3: compute the differentiation matrices
#     A_re = basics.assemble_A_re(freq_vec, tau_vec, epsilon, rbf_type)
#     A_re_R_inf = np.ones((N_freqs, 1))
#     A_re_L_0 = np.zeros((N_freqs, 1))
#     A_re = np.hstack(( A_re_R_inf, A_re_L_0, A_re))
#     A_im = basics.assemble_A_im(freq_vec, tau_vec, epsilon, rbf_type)
#     A_im_R_inf = np.zeros((N_freqs, 1))
#     A_im_L_0 = 2*np.pi*freq_vec.reshape((N_freqs, 1))
#     A_im = np.hstack(( A_im_R_inf, A_im_L_0, A_im))
#     A = np.vstack((A_re, A_im))
    
#     # step 4: compute the differentiation matrix
#     M2 = np.zeros((N_taus+2, N_taus+2))
#     M2[2:,2:] = basics.assemble_M_2(tau_vec, epsilon, rbf_type)
    
#     # step 5: compute the regularization parameter
# #     lambda_value = basics.optimal_lambda(A_re, A_im, np.real(Z_exp), np.imag(Z_exp), M2, -3, 'GCV')
#     lambda_value = 0.002
    
#     # step 6: recover the DRT
#     ### 
#     lb = np.zeros([N_taus+2])
#     bound_mat = np.eye(lb.shape[0])
#     H_combined, c_combined = basics.quad_format_combined(A_re, A_im, np.real(Z_exp_list[n]), np.imag(Z_exp_list[n]), M2, lambda_value)
    
#     # sol = solvers.qp(matrix(H_combined), matrix(c_combined),G,h)
#     ## deconvolved DRT
#     x = np.array(sol['x']).flatten()
    
#     R_inf_DRT_list[n], L_0_DRT_list[n] = x[0:2]
# #     print('Toto je x',x)
#     gamma_DRT_list[n] = np.array(x)[2:]
    
#     # step 7: recover the impedance
#     Z_DRT = A@x
#     Z_DRT_list[n] = Z_DRT[0:N_freqs] + 1j*Z_DRT[N_freqs:]

6
6
6
6
6
6
[-0.00152191+0.07008927j  0.00316502+0.06017453j  0.00661688+0.05177714j
  0.00865383+0.0443515j   0.01020544+0.03789467j  0.0114214 +0.03227731j
  0.01251879+0.02757261j  0.01339049+0.02380717j  0.01372384+0.02086611j
  0.01354676+0.01823659j  0.01318868+0.01565697j  0.01288893+0.01314927j
  0.01271746+0.01082661j  0.01262643+0.00873621j  0.0126472 +0.0068945j
  0.01266768+0.00514421j  0.01289822+0.00357306j  0.01313065+0.00220967j
  0.01340541+0.00096844j  0.0136539 -0.00016202j  0.01396897-0.00124172j
  0.01429837-0.00228318j  0.01469054-0.00324675j  0.01506989-0.00419803j
  0.01547511-0.00519439j  0.01593706-0.00617796j  0.01640694-0.00723161j
  0.01692572-0.00830461j  0.01741933-0.00945412j  0.01797909-0.01069921j
  0.01858433-0.01208224j  0.01928198-0.01364031j  0.02001899-0.01542678j
  0.02079267-0.01730531j  0.02163169-0.01944936j  0.02264486-0.0218933j
  0.02380797-0.02464592j  0.02507902-0.02774765j  0.02660533-0.03133419j
  0.02837841-0.03536124j  0.03040367-0.03

#### 2.2 Nyquist plots

In [29]:
cmap = plt.get_cmap('rainbow')
colors = cmap(np.linspace(0,1,N_exp)) 
# Rct = []
# Nyquist plots of the experimental and regressed impedances
for n in range(N_exp):
#     plt.plot(np.real(Z_exp_list[n]), -np.imag(Z_exp_list[n]), 'o', markersize=7, color='red')
    plt.plot(np.real(Z_exp_list[n]), -np.imag(Z_exp_list[n]), linewidth=4, color=colors[n], label= str(1.36+n*0.02))
plt.legend(frameon=False, fontsize = 15, loc='upper right')
plt.axis('scaled')
plt.xticks(np.arange(0, 300.1, 50))
plt.yticks(np.arange(-20, 200.1, 20))
plt.gca().set_aspect('equal', adjustable='box')
plt.xlabel(r'$Z_{\rm re}/\Omega$', fontsize = 20)
plt.ylabel(r'$-Z_{\rm im}/\Omega$', fontsize = 20)
fig = plt.gcf()
fig.set_size_inches(6.472, 4)
plt.show()

#### 2.3 DRT plots

In [64]:
# Plots of the recovered DRTs

cmap = plt.get_cmap('rainbow')
colors = cmap(np.linspace(0,1,len(gamma_DRT_list))) 
for n in range(0,len(gamma_DRT_list)):
    plt.plot(log_tau_list[n], gamma_DRT_list[n], linewidth=4, color=colors[n])
# plt.legend(frameon=False, fontsize = 15)
# plt.axis([1E-5, 1E0, 0, 1])
# plt.xlabel(r'$\tau/\rm s$', fontsize = 20)
# plt.ylabel(r'$\gamma/\Omega$', fontsize = 20)
# fig.set_size_inches(6.472, 4)
#plt.savefig('2ZARC_DRT-plots.svg', dpi=300, bbox_inches='tight') # save the picture
#plt.savefig('2ZARC_DRT-plots.pdf', dpi=300, bbox_inches='tight')
plt.show()

#### 2.4 Contour plot of the recovered DRTs

In [23]:
# Assuming these variables are defined somewhere in your code
N_exp = 10  # Number of experimental conditions

# Custom ticks for the experimental conditions
custom_ticks = [1,2,3,4,5,6,7,8,9,10]

# Generate the necessary matrices for contour plotting
T_range = np.arange(N_exp)  # Experimental condition that varies between one EIS measurement to another
temp_vec = np.array(custom_ticks, dtype=float)  # Use custom ticks as the temperature vector
tau_mat, temp_mat = np.meshgrid(tau_vec, temp_vec)
gamma_norm_global_mat = np.zeros((len(custom_ticks), len(tau_vec)))

for index, temp in enumerate(T_range):
    gamma_norm_global_mat[index, :] = gamma_DRT_list[index]

fig = plt.figure(figsize=(6.472, 4)) 
gs = gridspec.GridSpec(1, 1) 
plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=0.5)
ax = plt.subplot(gs[0, 0])
cs = ax.contourf(tau_mat, temp_mat, gamma_norm_global_mat, cmap=plt.cm.plasma)
fig.colorbar(cs, label=r'$\gamma/\Omega$')

# Set custom ticks for the y-axis (experimental conditions)
ax.set_yticks(custom_ticks)

ax.set_ylabel(r'$R_{\rm ct}/\Omega$', fontsize=20)
ax.set_xlabel(r'$\tau/s$', fontsize=20)
ax.set_xscale('log')
fig = plt.gcf()
fig.set_size_inches(6.472, 4)

# plt.savefig('figs/2ZARC_contour-plot.svg', dpi=300, bbox_inches='tight')  # save the picture
# plt.savefig('figs/2ZARC_contour-plot.pdf', dpi=300, bbox_inches='tight')
plt.show()

#### 2.5 Save the recovered DRTs and impedances

In [200]:
for n in range(N_exp):
    
    # save the DRT recovered with RR
    gamma_DRT_excel = pd.DataFrame(gamma_DRT_list[n])
    gamma_DRT_excel.to_csv('./results/2ZARC_DRT-RR_'+str(n)+'.csv',index=False)

    # save the impedance recovered with RR
    df = pd.DataFrame.from_dict({'Freq': freq_vec, 'Real': np.real(Z_DRT_list[n]), 'Imag': np.imag(Z_DRT_list[n])})
    df.to_csv('./results/2ZARC_Z-RR_'+str(n)+'.csv')

#### 2.6 Import the recovered DRTs and impedances

In [201]:
gamma_DRT_list = [0]*N_exp
Z_DRT_list = [0]*N_exp

for n in range(N_exp):
    
    # import the recovered DRTs
    file = open('./results/2ZARC_DRT-RR_'+str(n)+'.csv')
    gamma_DRT_list[n] = loadtxt(file, delimiter = ",")[1:] # because there's a 0 in the first position

    # import the recovered impedances
    df = pd.read_csv('./results/2ZARC_Z-RR_'+str(n)+'.csv') # this artificial data was generated using the 2xZARC model
    Z_DRT = np.flip(df['Real'].values + 1j*df['Imag'].values)
    Z_DRT_list[n] = Z_exp